# 05.02 · MCP fundamentals: hosts, clients, servers, primitives, transports

**Module 05 · MCP & External Tools** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lesson 02.01 (tools). Module 05.01 helps but isn't required. |
| 🔑 Keys | none (no model calls in this lesson) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** the MCP roles (host, client, server) and the three server primitives (tools, resources, prompts).
2. **Use** top-level `await` in Jupyter, and explain why `asyncio.run()` fails there.
3. **Inspect** a server: its info, capabilities, tools (with schemas and annotations), resources and prompts.
4. **Compare** the transports: in-memory, stdio and Streamable HTTP, and know which works where (Windows included).
5. **Recognize** `langchain.mcp` as beta and what that means for you.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [MCP: Architecture overview](https://modelcontextprotocol.io/docs/learn/architecture) | Hosts, clients, servers; data and transport layers |
| [MCP: Server concepts](https://modelcontextprotocol.io/docs/learn/server-concepts) | Tools (model-controlled), resources (app-controlled), prompts (user-controlled) |
| [MCP specification: Transports](https://modelcontextprotocol.io/specification/latest/basic/transports) | stdio and Streamable HTTP |
| [LangChain: Model Context Protocol](https://docs.langchain.com/oss/python/langchain/mcp/index) | `langchain.mcp` (beta) overview |
| [FastMCP: Client](https://gofastmcp.com/clients/client) | The client we use to look inside servers |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain[mcp]>=1.4.4,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "fastmcp", "mcp",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/05-mcp-and-external-tools/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import logging
import warnings

# (a) Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# (b) `langchain.mcp` is beta and says so once per process (LangChainBetaWarning). Read it once in 05.02; muted after.
from langchain_core._api import LangChainBetaWarning  # noqa: E402  (import first: langchain_core adds its own filter)

warnings.filterwarnings("ignore", message=".*langchain.mcp.* is in beta", category=LangChainBetaWarning)
# (c) In-process FastMCP servers log every failing tool call with a long boxed traceback. The client still gets
#     the error as a ToolMessage(status="error"); we show it ourselves. Remove these lines to see the server's view.
import fastmcp  # noqa: E402,F401  (importing fastmcp configures its logger, so set the level after the import)

logging.getLogger("fastmcp").setLevel(logging.CRITICAL)

In [ ]:
# Step 5: Imports. The notes server is this module's FastMCP server (you'll build your own in 05.04).
import asyncio
import subprocess
import sys
import tempfile
import textwrap

from fastmcp import Client

from mastery.mcp import serve_http
from mastery.mcp_servers.notes_server import build_server

NOTES_DIR = tempfile.mkdtemp(prefix="m05-notes-")   # A throwaway notes folder for this lesson.
notes_server = build_server(NOTES_DIR)
print("server:", notes_server.name)

**You should now see** `server: course-notes`.

---
## Part 1 · The roles

```
 Host (your app: Claude Desktop, VS Code, Cursor, or your LangChain agent)
   ├── MCP client ──(transport)──▶ MCP server A  (e.g. LangChain docs)
   └── MCP client ──(transport)──▶ MCP server B  (e.g. your notes server)
```

- **Host**: the application the user talks to. It decides which servers to connect and what the model may use.
- **Client**: one connection to one server, inside the host. It speaks JSON-RPC 2.0.
- **Server**: a program that exposes capabilities. It can run on your machine (stdio, local HTTP) or anywhere (remote HTTP).

A server offers three **primitives**, each controlled by someone different:

| Primitive | Controlled by | Example from our notes server |
|---|---|---|
| **Tools** | The model (it decides to call them) | `save_note`, `search_notes`, `delete_note` |
| **Resources** | The application (it decides what to read into context) | `notes://{note_id}`: the text of one note |
| **Prompts** | The user (they pick a template) | `study_plan(topic)` |

Clients can offer features back to servers too: **elicitation** (the server asks the user a question), **sampling** (the server asks the host's model), **roots** (which folders the server may see). You'll meet elicitation in 05.05.

---
## Part 2 · Jupyter skill: top-level `await`

MCP clients are **async**: every call waits on a network or a process. In a script you write `asyncio.run(main())`. In Jupyter an event loop is **already running**, so `asyncio.run` fails. Instead, write `await` directly in a cell.

In [ ]:
# 🧪 Offline: asyncio.run() inside Jupyter fails; a top-level await works.
async def two_plus_two() -> int:
    await asyncio.sleep(0.1)
    return 4

coro = two_plus_two()
try:
    asyncio.run(coro)
except RuntimeError as exc:
    coro.close()   # Tidy up the coroutine that never ran (avoids a "never awaited" warning).
    print("asyncio.run →", exc)

print("await       →", await two_plus_two())

**You should now see** `asyncio.run → asyncio.run() cannot be called from a running event loop` and `await → 4`.

Rules for this module: in notebooks, `await` at the top level; in `.py` scripts, `asyncio.run(main())`. Don't install `nest_asyncio` to "fix" it: it patches the loop globally and hides real bugs.

---
## Part 3 · Look inside a server (in-memory transport)

`fastmcp.Client(server_object)` connects to a server **in the same process**: no subprocess, no port. It's the fastest way to explore, and it behaves the same on Windows, macOS and Linux. `async with` opens the session (the `initialize` handshake) and closes it at the end.

In [ ]:
# 🧪 Offline: the handshake result, then each primitive.
async with Client(notes_server) as client:
    print("server  :", client.server_info.name, client.server_info.version)
    print("protocol:", client.protocol_version)
    print("offers  :", [name for name in ("tools", "resources", "prompts") if getattr(client.server_capabilities, name)])
    print("instructions:", client.instructions)
    tools = await client.list_tools()
    templates = await client.list_resource_templates()
    prompts = await client.list_prompts()

print()
for t in tools:
    hints = t.annotations
    print(f"🔧 {t.name:<13} read_only={hints.read_only_hint!s:<5} destructive={hints.destructive_hint!s:<5} args={list(t.input_schema['properties'])}")
for r in templates:
    print(f"📄 {r.uri_template}  ({r.mime_type})  {r.description}")
for p in prompts:
    print(f"💬 {p.name}({', '.join(a.name for a in p.arguments or [])})  {p.description}")

**You should now see** the server name and version, `protocol: 2026-07-28`, `offers: ['tools', 'resources', 'prompts']`, the server's instructions, five 🔧 tools with their hints and arguments (`save_note` not read-only, `delete_note` destructive), one 📄 template `notes://{note_id}`, and one 💬 prompt `study_plan(topic)`.

The **annotations** (`read_only_hint`, `destructive_hint`, ...) are *claims made by the server*. Useful for your own servers; for third-party servers, don't trust them blindly (05.05).

In [ ]:
# 🧪 Offline: call a tool, read a resource, render a prompt. Errors come back as results, not crashes.
async with Client(notes_server) as client:
    saved = await client.call_tool("save_note", {"title": "Top-level await", "body": "Use await in Jupyter, asyncio.run in scripts.",
                                                 "tags": ["async"]})
    print("tool     →", saved.data)
    contents = await client.read_resource(f"notes://{saved.data['id']}")
    print("resource →", contents[0].text.replace("\n", " ⏎ "))
    prompt = await client.get_prompt("study_plan", {"topic": "async"})
    print("prompt   →", prompt.messages[0].content.text[:90], "...")
    bad = await client.call_tool("save_note", {"title": "", "body": ""}, raise_on_error=False)
    print("error    →", bad.is_error, bad.content[0].text)

**You should now see** `tool → {'id': 'n-...', 'title': 'Top-level await', 'tags': ['async']}`, the note text from the resource, the start of the rendered prompt, and `error → True Error calling tool 'save_note': A note needs a non-empty title and body.`

### What just happened?

Under the hood each line was a JSON-RPC message: `initialize`, `tools/list`, `tools/call`, `resources/read`, `prompts/get`. The server answered a bad call with a result marked `isError: true`, so the model can read the error and try again. That's the MCP convention: tool errors are data; only protocol or transport failures raise.

---
## Part 4 · Transports

| Transport | How it runs | Use it for | Windows notes |
|---|---|---|---|
| **In-memory** | Same Python process | Tests, notebooks, exploration | Works everywhere |
| **stdio** | Host starts the server as a **subprocess**, talks over stdin/stdout | Local servers in IDEs and desktop apps | Fine from a script or an IDE. From *inside Jupyter on Windows* it may fail (see Pitfalls) |
| **Streamable HTTP** | Server listens on a URL (`http://127.0.0.1:8770/mcp`) | Remote servers, shared servers, auth | Bind to `127.0.0.1` to avoid the firewall prompt |
| SSE | Older HTTP style | Legacy servers only | Deprecated in the spec |

**Streamable HTTP, locally:** `serve_http()` starts the notes server in a separate process (`python -m mastery.mcp_servers.notes_server --transport http ...`) on a free port, yields its URL, and stops it on exit.

In [ ]:
# 🧪 Offline (local HTTP): the same server, now over the network stack.
with serve_http(env={"MASTERY_NOTES_DIR": NOTES_DIR}) as url:
    print("url:", url)
    async with Client(url) as client:
        print("tools over HTTP:", [t.name for t in await client.list_tools()])
        found = await client.call_tool("search_notes", {"query": "jupyter await"})
        print("search:", found.structured_content)
print("server stopped")

**You should now see** `url: http://127.0.0.1:<port>/mcp`, the same five tools, the note you saved in Part 3 (the HTTP server reads the same folder), and `server stopped`.

**stdio:** the client starts `python notes_server.py` itself. We run that from a small **script** in a fresh process, the way an IDE would, which also works on Windows.

In [ ]:
# 🧪 Offline (stdio): a child script launches the server as a subprocess and lists its tools.
child = textwrap.dedent(f"""
    import asyncio, sys
    from fastmcp import Client
    from fastmcp.client.transports import StdioTransport

    async def main():
        transport = StdioTransport(command=sys.executable, args=["-m", "mastery.mcp_servers.notes_server"],
                                   env={{"MASTERY_NOTES_DIR": {NOTES_DIR!r}, "PYTHONPATH": {str(REPO_ROOT / "src")!r}}})
        async with Client(transport) as client:
            print("stdio tools:", [t.name for t in await client.list_tools()])

    asyncio.run(main())   # A script: asyncio.run is right here.
""")
done = subprocess.run([sys.executable, "-c", child], capture_output=True, text=True, encoding="utf-8", timeout=60)
print(done.stdout.strip() or done.stderr.strip()[-600:])

**You should now see** `stdio tools: ['save_note', 'search_notes', 'list_notes', 'delete_note', 'lesson_info']`. If you see a traceback instead, read its last line; the Pitfalls table covers the usual ones.

One rule matters for stdio servers: **never `print()` to stdout** in a stdio server. Stdout *is* the protocol channel; a stray print corrupts it. Log to stderr.

---
## Part 5 · A real remote server, and `langchain.mcp` is beta

> 🌐 Needs internet, no key, no tokens. Offline or behind a firewall? The cell prints a ⚠️ and the lesson continues with the offline stand-in.

In [ ]:
# 🌐 The official LangChain docs MCP server: no key, read-only search over docs.langchain.com.
DOCS_MCP = "https://docs.langchain.com/mcp"
try:
    async with Client(DOCS_MCP, timeout=20) as client:
        print("server:", client.server_info.name, "| protocol:", client.protocol_version)
        for t in await client.list_tools():
            print(f"🔧 {t.name}: {(t.description or '').splitlines()[0][:90]}")
except Exception as exc:
    print(f"⚠️ Could not reach {DOCS_MCP} ({type(exc).__name__}: {str(exc)[:120]}). Offline? 05.03 has a stand-in.")

**You should now see** the server name and two or three tools, among them `search_docs_by_lang_chain` (search the docs) and `query_docs_filesystem_docs_by_lang_chain` (browse pages like files). Or a ⚠️ line if you are offline.

In [ ]:
# 🧪 Offline: langchain.mcp announces it is beta, once per process.
import warnings

from langchain_core._api import LangChainBetaWarning

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    import importlib
    import langchain.mcp
    importlib.reload(langchain.mcp)   # Re-import so the warning fires even if something imported it earlier.
beta = [w for w in caught if issubclass(w.category, LangChainBetaWarning)]
print(beta[0].message if beta else "no beta warning (has langchain.mcp gone stable? check the research checkpoint)")
print("exports:", sorted(n for n in langchain.mcp.__all__))

**You should now see** a `LangChainBetaWarning` message saying `langchain.mcp` is in beta, and `exports: ['MCPAdapter', 'MCPToolArtifact', 'as_langchain_tool']` (plus the submodule `langchain.mcp.elicitation`, used in 05.05).

**Beta** means: works and is supported, but names and behaviour may change in a minor release. The course pins `langchain>=1.4.4,<2`, wraps MCP code in `mastery/mcp.py`, and records what it relies on (decision log D-013). After this cell the warning is muted (Step 4).

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `RuntimeError: asyncio.run() cannot be called from a running event loop` | `asyncio.run` in Jupyter | Use top-level `await` |
| `SyntaxError: 'await' outside function` | Running the code as a `.py` script | Wrap it in `async def main()` + `asyncio.run(main())` |
| `NotImplementedError` from `asyncio` subprocess when using stdio **in Jupyter on Windows** | Some Jupyter setups on Windows run a loop that can't start subprocesses | Use in-memory or HTTP (`serve_http`) in notebooks; use stdio from scripts and IDEs |
| Windows Firewall pop-up when starting a server | Binding to `0.0.0.0` | Bind to `127.0.0.1` (the course default) |
| `localhost` is slow or refused | `localhost` → IPv6 `::1` first | Use `http://127.0.0.1:...` |
| stdio client hangs or gets garbage | The server prints to stdout | Log to stderr; never `print` in stdio servers |
| `Address already in use` | Port taken (e.g. by the course mock on 8765) | `serve_http()` picks a free port; notes server default is 8770 |
| `UnicodeEncodeError` with `ñ`/`é` on Windows | Console code page | The course sets `PYTHONIOENCODING=utf-8` for child servers; files use UTF-8 |
| `ConnectError` / SSL errors to `docs.langchain.com` | Offline or corporate proxy | [00.02 Step 4b](../00-foundations/00-02-create-course-env.md#troubleshooting) (`SSL_CERT_FILE`) |

## ⚠️ Production notes

> ⚠️ **Production note: an MCP server is code you run with your permissions.**
> A stdio server runs as a local process with your user's rights (files, network, credentials in the environment). Install servers like you install packages: from sources you trust, pinned to a version, with the minimum environment variables passed in.

> ⚠️ **Production note: beta APIs need a wrapper.**
> Keep `langchain.mcp` calls in one module (`mastery/mcp.py` here) and pin the version, so a breaking change is a one-file fix.

## 🏋️ Exercises

1. **(Easy)** Call `lesson_info` with `"05.02"` through the in-memory client. What fields come back?
2. **(Medium)** Save three notes, then `read_resource` each one by URI. What happens with `notes://n-doesnotexist`?
3. **(Medium)** Run the stdio child script again, but add `print("hello")` to the top of a copy of the server. What does the client do?
4. **(Stretch)** If you have Node.js: `npx @modelcontextprotocol/inspector`, choose *Streamable HTTP*, and connect to a server started with `python -m mastery.mcp_servers.notes_server --transport http` (from the repo root, env active, `PYTHONPATH=src`).

<details>
<summary><b>Show solutions</b></summary>

**1.** `await client.call_tool("lesson_info", {"lesson": "05.02"})` → `title`, `objectives`, `reading` (URLs) parsed from `COURSE_OUTLINE.md`.

**2.** Each URI returns the note text. An unknown id raises a `ResourceError`-style `McpError` ("Unknown note ..."): resources fail as errors, unlike tool results.

**3.** The client sees a line that isn't JSON-RPC and logs a parse error or fails the handshake. That's why stdio servers log to stderr.

**4.** The Inspector shows the same tools, resources and prompts in a web UI, and lets you call them by hand. Great for debugging a server before an agent uses it. In Git Bash: `PYTHONPATH=src python -m mastery.mcp_servers.notes_server --transport http`.
</details>

## ✅ Checkpoint

- [ ] I can name the three roles and the three server primitives, and who controls each primitive.
- [ ] I use top-level `await` in notebooks and `asyncio.run` in scripts.
- [ ] I can list a server's tools (with annotations), resources and prompts.
- [ ] I can pick a transport and know the Windows caveats.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Is `langchain.mcp` still beta (does the cell above still print `LangChainBetaWarning`)? Which features are still **not yet wrapped**? See "Prompts and resources" and "Deprecated in the MCP protocol" on the [migration page](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters) (as of 2026-10-12: prompts and resources have no wrapper, so use the FastMCP client's `get_prompt` / `read_resource` as in Part 3; SSE is deprecated by the protocol). Log the answer in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Host → client → server; servers offer tools (model), resources (app) and prompts (user).
- MCP is async: top-level `await` in notebooks, `asyncio.run` in scripts.
- Tool errors are results (`isError`); transports are in-memory, stdio, Streamable HTTP.
- `langchain.mcp` is beta: wrap it and pin it.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 05.03 · Consuming MCP servers with MCPAdapter** → [`05-03-mcp-adapter.ipynb`](05-03-mcp-adapter.ipynb).

[Module 05 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)